# MLOps Pipeline: Fraud Detection in Snowflake
*Co-authored with CoCo*

End-to-end MLOps lifecycle for fraud detection using scikit-learn/XGBoost — from EDA through model serving — all within Snowflake.

**Sections:**
1. Exploratory Data Analysis
2. Feature Engineering
3. Model Training & Evaluation
4. Model Registry & Warehouse Inference
5. Batch Scoring
6. Model Monitoring & Drift Detection (automated with Tasks + Alerts)

---
## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE DATABASE COCO_DS_DEMO").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE COCO_DS_WH").collect()
print(f"Connected | Role: {session.get_current_role()} | DB: {session.get_current_database()}")

---
## 1. Exploratory Data Analysis

Load the TRANSACTIONS table and explore fraud patterns across merchant categories, payment methods, channels, amounts, and time-of-day.

In [ ]:
# Load full dataset into pandas
df = session.table("TRANSACTIONS").to_pandas()
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['IS_FRAUD'].mean():.4f} ({df['IS_FRAUD'].sum():,} / {len(df):,})")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nColumn types:")
print(df.dtypes.to_string())

In [ ]:
# Fraud rate by merchant_category, payment_method, and channel
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ['MERCHANT_CATEGORY', 'PAYMENT_METHOD', 'CHANNEL']):
    rates = df.groupby(col)['IS_FRAUD'].mean().sort_values(ascending=False)
    rates.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Fraud Rate by {col}')
    ax.set_xlabel('Fraud Rate')
    ax.axvline(df['IS_FRAUD'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Amount distribution: fraud vs legit
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color in [(True, 'coral'), (False, 'steelblue')]:
    subset = df[df['IS_FRAUD'] == label]['AMOUNT']
    axes[0].hist(subset, bins=60, alpha=0.6, label=f"{'Fraud' if label else 'Legit'}", color=color, density=True)
axes[0].set_title('Amount Distribution (Fraud vs Legit)')
axes[0].set_xlabel('Amount')
axes[0].legend()
axes[0].set_xlim(0, 2000)

# Fraud rate by hour_of_day
hourly = df.groupby('HOUR_OF_DAY')['IS_FRAUD'].mean()
axes[1].bar(hourly.index, hourly.values, color='darkorange')
axes[1].axhline(df['IS_FRAUD'].mean(), color='red', linestyle='--', label='Overall')
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Fraud Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fraud vs is_international + correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# International fraud comparison
intl = df.groupby('IS_INTERNATIONAL')['IS_FRAUD'].mean()
axes[0].bar(['Domestic', 'International'], intl.values, color=['steelblue', 'coral'])
axes[0].set_title('Fraud Rate: Domestic vs International')
axes[0].set_ylabel('Fraud Rate')
for i, v in enumerate(intl.values):
    axes[0].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

# Correlation heatmap
numeric_cols = ['AMOUNT', 'HOUR_OF_DAY', 'IS_INTERNATIONAL', 'IS_FRAUD']
corr_df = df[numeric_cols].copy()
corr_df['IS_INTERNATIONAL'] = corr_df['IS_INTERNATIONAL'].astype(int)
corr_df['IS_FRAUD'] = corr_df['IS_FRAUD'].astype(int)
sns.heatmap(corr_df.corr(), annot=True, cmap='RdBu_r', center=0, ax=axes[1], fmt='.3f')
axes[1].set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

---
## 2. Feature Engineering

Create predictive features from raw transaction data:
- Statistical features (z-score, high-amount flag)
- Temporal features (odd-hour flag)
- Target encoding (category/channel risk scores)
- One-hot encoding of categoricals

In [ ]:
df_feat = df.copy()

# AMOUNT_ZSCORE
amount_mean = df_feat['AMOUNT'].mean()
amount_std = df_feat['AMOUNT'].std()
df_feat['AMOUNT_ZSCORE'] = (df_feat['AMOUNT'] - amount_mean) / amount_std

# IS_HIGH_AMOUNT: > 2 std above mean
df_feat['IS_HIGH_AMOUNT'] = (df_feat['AMOUNT'] > amount_mean + 2 * amount_std).astype(int)

# IS_ODD_HOUR: hour between 1-5
df_feat['IS_ODD_HOUR'] = df_feat['HOUR_OF_DAY'].between(1, 5).astype(int)

# CATEGORY_RISK_SCORE: fraud rate per merchant_category
category_risk = df_feat.groupby('MERCHANT_CATEGORY')['IS_FRAUD'].mean()
df_feat['CATEGORY_RISK_SCORE'] = df_feat['MERCHANT_CATEGORY'].map(category_risk)

# CHANNEL_RISK_SCORE: fraud rate per channel
channel_risk = df_feat.groupby('CHANNEL')['IS_FRAUD'].mean()
df_feat['CHANNEL_RISK_SCORE'] = df_feat['CHANNEL'].map(channel_risk)

# One-hot encode categoricals
df_feat = pd.get_dummies(df_feat, columns=['MERCHANT_CATEGORY', 'PAYMENT_METHOD', 'CHANNEL'], dtype=int)

# Convert boolean
df_feat['IS_INTERNATIONAL'] = df_feat['IS_INTERNATIONAL'].astype(int)
df_feat['IS_FRAUD'] = df_feat['IS_FRAUD'].astype(int)

# Drop non-feature columns
drop_cols = ['TRANSACTION_ID', 'CUSTOMER_ID', 'TRANSACTION_DATE', 'LOCATION_COUNTRY', 'AMOUNT', 'HOUR_OF_DAY']
df_model = df_feat.drop(columns=drop_cols)

print(f"Model-ready dataset: {df_model.shape}")
print(f"\nFeatures ({df_model.shape[1] - 1}):")
feature_cols = [c for c in df_model.columns if c != 'IS_FRAUD']
for c in feature_cols:
    print(f"  {c}")

---
## 3. Model Training & Evaluation

Compare three classifiers on stratified 70/30 split:
- **XGBClassifier** with `scale_pos_weight` for class imbalance
- **RandomForestClassifier** with `class_weight='balanced'`
- **LogisticRegression** as linear baseline

Evaluate: Accuracy, Precision, Recall, F1, AUC-ROC

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Prepare X, y
X = df_model[feature_cols]
y = df_model['IS_FRAUD']

# 70/30 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Fraud in train: {y_train.mean():.4f} | scale_pos_weight: {scale_pos:.1f}")

In [ ]:
# Train three models
models = {
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos, eval_metric='logloss',
        random_state=42, verbosity=0
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_proba),
        'y_pred': y_pred,
        'y_proba': y_proba
    }

# Display metrics table
metrics_df = pd.DataFrame({k: {m: v for m, v in r.items() if m not in ['model','y_pred','y_proba']}
                           for k, r in results.items()}).T
print(metrics_df.round(4).to_string())

# Select best model
best_name = metrics_df['f1'].idxmax()
best_model = results[best_name]['model']
print(f"\n>>> Best model: {best_name} (F1={results[best_name]['f1']:.4f}, AUC={results[best_name]['auc_roc']:.4f})")

In [ ]:
# ROC curves + confusion matrix for best model
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'XGBoost': 'coral', 'Random Forest': 'steelblue', 'Logistic Regression': 'green'}
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['auc_roc']:.3f})", color=colors[name], lw=2)
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[0].set_title('ROC Curves')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Confusion matrix
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=axes[1],
            xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
axes[1].set_title(f'Confusion Matrix ({best_name})')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (top 10)
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols)
else:
    imp = pd.Series(np.abs(best_model.coef_[0]), index=feature_cols)

top10 = imp.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 4))
top10.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Top 10 Feature Importances ({best_name})')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 4. Model Registry & Warehouse Inference

Register the best model in Snowflake's Model Registry and demonstrate warehouse-based inference.

**Key points:**
- `target_platforms=["WAREHOUSE"]` — model runs on the warehouse, no container service needed
- `mv.run(data, function_name="predict_proba")` — invoke predictions directly
- `create_service()` is for SPCS container deployments only (requires compute pool)

In [ ]:
from snowflake.ml.registry import Registry

# Initialize registry
reg = Registry(session=session, database_name="COCO_DS_DEMO", schema_name="PUBLIC")

# Prepare metrics and sample data
training_metrics = {
    'accuracy': float(results[best_name]['accuracy']),
    'precision': float(results[best_name]['precision']),
    'recall': float(results[best_name]['recall']),
    'f1_score': float(results[best_name]['f1']),
    'auc_roc': float(results[best_name]['auc_roc'])
}
sample_input = X_test.head(10)

# Log model to registry
mv = reg.log_model(
    model=best_model,
    model_name="fraud_detector",
    version_name="v1",
    metrics=training_metrics,
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"]
)

print(f"Model registered: fraud_detector v1")
print(f"Metrics: {training_metrics}")

In [ ]:
# List registered models and versions
print("Registered models:")
print(reg.show_models()[['name', 'default_version_name']].to_string())

model_ref = reg.get_model("fraud_detector")
print(f"\nVersions for fraud_detector:")
print(model_ref.show_versions()[['name', 'is_default_version']].to_string())

In [ ]:
# Warehouse-based inference demo
sample = X_test.head(5)
predictions = mv.run(sample, function_name="predict_proba")
print("Warehouse inference (predict_proba):")
print(predictions)

print("\n--- SQL equivalent ---")
print("SELECT COCO_DS_DEMO.PUBLIC.FRAUD_DETECTOR!PREDICT_PROBA(...)")
print("-- Pass feature columns as function arguments")
print("\nNote: create_service() is for SPCS container deployments only.")
print("Warehouse inference uses mv.run() or MODEL!FUNCTION() SQL syntax.")

---
## 5. Batch Scoring

Score all transactions, write predictions to Snowflake, and analyze risk tiers.

In [ ]:
# Score all transactions
all_proba = best_model.predict_proba(X)[:, 1]

predictions_df = pd.DataFrame({
    'TRANSACTION_ID': df['TRANSACTION_ID'],
    'FRAUD_PROBABILITY': np.round(all_proba, 6),
    'PREDICTED_FRAUD': (all_proba >= 0.5).astype(bool),
    'SCORED_AT': datetime.now()
})

# Write to Snowflake
session.create_dataframe(predictions_df).write.mode("overwrite").save_as_table(
    "COCO_DS_DEMO.PUBLIC.FRAUD_PREDICTIONS"
)
print(f"Predictions written: {len(predictions_df):,} rows")
print(f"Predicted fraud: {predictions_df['PREDICTED_FRAUD'].sum():,}")

In [ ]:
# Risk tier summary
predictions_df['RISK_TIER'] = pd.cut(
    predictions_df['FRAUD_PROBABILITY'],
    bins=[0, 0.4, 0.8, 1.0],
    labels=['Low (<0.4)', 'Medium (0.4-0.8)', 'High (>0.8)'],
    include_lowest=True
)

tier_summary = predictions_df.groupby('RISK_TIER', observed=True).agg(
    count=('TRANSACTION_ID', 'count'),
    avg_prob=('FRAUD_PROBABILITY', 'mean')
).reset_index()
print("Risk Tier Summary:")
print(tier_summary.to_string(index=False))

# Top 20 most suspicious
print(f"\nTop 20 Most Suspicious Transactions:")
top20 = predictions_df.nlargest(20, 'FRAUD_PROBABILITY')[['TRANSACTION_ID', 'FRAUD_PROBABILITY']]
print(top20.to_string(index=False))

---
## 6. Model Monitoring & Drift Detection

Simulate production monitoring:
1. Generate a drifted batch (5% fraud, higher amounts, more digital_wallet)
2. Score and measure performance degradation
3. Detect **data drift** using KS test per feature
4. Detect **prediction drift** (score distribution shift)
5. Log results to monitoring table
6. Create automated **Snowflake Task** and **Alert** for daily monitoring

In [ ]:
from scipy import stats

# Simulate drifted batch: 5% fraud, higher amounts, more digital_wallet
np.random.seed(99)
n_new = 50000

new_batch = pd.DataFrame({
    'IS_FRAUD': np.random.choice([0, 1], n_new, p=[0.95, 0.05]),
})

# Amount: shift higher
new_batch['AMOUNT_ZSCORE'] = np.where(
    new_batch['IS_FRAUD'] == 1,
    np.random.normal(5.5, 3, n_new),  # fraud: higher z-score
    np.random.normal(0.3, 1.2, n_new)  # legit: slight upward shift
)
new_batch['IS_HIGH_AMOUNT'] = (new_batch['AMOUNT_ZSCORE'] > 2).astype(int)
new_batch['IS_ODD_HOUR'] = np.where(
    new_batch['IS_FRAUD'] == 1,
    np.random.binomial(1, 0.6, n_new),
    np.random.binomial(1, 0.2, n_new)
)
new_batch['CATEGORY_RISK_SCORE'] = np.where(
    new_batch['IS_FRAUD'] == 1,
    np.random.uniform(0.04, 0.08, n_new),
    np.random.uniform(0.02, 0.04, n_new)
)
new_batch['CHANNEL_RISK_SCORE'] = np.random.uniform(0.025, 0.035, n_new)
new_batch['IS_INTERNATIONAL'] = np.where(
    new_batch['IS_FRAUD'] == 1,
    np.random.binomial(1, 0.55, n_new),
    np.random.binomial(1, 0.15, n_new)  # slightly more international
)

# One-hot: shift toward digital_wallet
for col in [c for c in feature_cols if c.startswith('MERCHANT_CATEGORY_')]:
    new_batch[col] = np.random.binomial(1, 0.17, n_new)
for col in [c for c in feature_cols if c.startswith('PAYMENT_METHOD_')]:
    new_batch[col] = 0
new_batch['PAYMENT_METHOD_digital_wallet'] = np.random.binomial(1, 0.45, n_new)  # shifted up
new_batch['PAYMENT_METHOD_credit_card'] = np.random.binomial(1, 0.25, n_new)
new_batch['PAYMENT_METHOD_debit_card'] = np.random.binomial(1, 0.20, n_new)
new_batch['PAYMENT_METHOD_wire'] = np.random.binomial(1, 0.10, n_new)
for col in [c for c in feature_cols if c.startswith('CHANNEL_')]:
    new_batch[col] = 0
new_batch['CHANNEL_online'] = np.random.binomial(1, 0.55, n_new)
new_batch['CHANNEL_in_store'] = np.random.binomial(1, 0.30, n_new)
new_batch['CHANNEL_atm'] = np.random.binomial(1, 0.15, n_new)

# Ensure all feature columns exist
for col in feature_cols:
    if col not in new_batch.columns:
        new_batch[col] = 0

X_new = new_batch[feature_cols]
y_new = new_batch['IS_FRAUD']

print(f"New batch: {n_new:,} transactions")
print(f"Fraud rate: {y_new.mean():.4f} (shifted from ~0.03 to ~0.05)")
print(f"Digital wallet share: {new_batch['PAYMENT_METHOD_digital_wallet'].mean():.3f} (shifted up)")

In [ ]:
# Score new batch and evaluate performance
new_proba = best_model.predict_proba(X_new)[:, 1]
new_pred = (new_proba >= 0.5).astype(int)

new_metrics = {
    'precision': precision_score(y_new, new_pred),
    'recall': recall_score(y_new, new_pred),
    'f1': f1_score(y_new, new_pred),
    'auc_roc': roc_auc_score(y_new, new_proba)
}

f1_threshold = 0.70

print("PERFORMANCE MONITORING:")
print(f"{'Metric':<12} {'Training':>10} {'Current':>10} {'Delta':>10} {'Alert':>6}")
print("-" * 52)
for metric in ['precision', 'recall', 'f1', 'auc_roc']:
    train_val = training_metrics.get(metric, training_metrics.get('f1_score'))
    if metric == 'f1': train_val = training_metrics['f1_score']
    curr_val = new_metrics[metric]
    delta = curr_val - train_val
    alert = '⚠️' if (metric == 'f1' and curr_val < f1_threshold) else ''
    print(f"{metric:<12} {train_val:>10.4f} {curr_val:>10.4f} {delta:>+10.4f} {alert:>6}")

In [ ]:
# DATA DRIFT: KS test per feature
drift_features = ['AMOUNT_ZSCORE', 'IS_HIGH_AMOUNT', 'IS_ODD_HOUR',
                  'CATEGORY_RISK_SCORE', 'CHANNEL_RISK_SCORE', 'IS_INTERNATIONAL',
                  'PAYMENT_METHOD_digital_wallet', 'CHANNEL_online']

drift_results = []
for feat in drift_features:
    ks_stat, ks_pval = stats.ks_2samp(X_train[feat].values, X_new[feat].values)
    drift_results.append({'feature': feat, 'ks_stat': ks_stat, 'p_value': ks_pval,
                          'drift_detected': ks_pval < 0.01})

drift_df = pd.DataFrame(drift_results)
print("DATA DRIFT (KS Test):")
print(drift_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['coral' if d else 'steelblue' for d in drift_df['drift_detected']]
ax.barh(drift_df['feature'], drift_df['ks_stat'], color=colors)
ax.axvline(0.05, color='red', linestyle='--', label='Threshold')
ax.set_xlabel('KS Statistic')
ax.set_title('Data Drift Detection (KS Test per Feature)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PREDICTION DRIFT: compare training vs new batch score distributions
fig, ax = plt.subplots(figsize=(10, 4))
train_proba = best_model.predict_proba(X_test)[:, 1]
ax.hist(train_proba, bins=50, alpha=0.6, label='Training (test set)', density=True, color='steelblue')
ax.hist(new_proba, bins=50, alpha=0.6, label='New Batch', density=True, color='coral')
ax.set_title('Prediction Drift: Score Distribution Shift')
ax.set_xlabel('Fraud Probability')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

pred_ks, pred_pval = stats.ks_2samp(train_proba, new_proba)
print(f"Prediction drift KS: {pred_ks:.4f} (p={pred_pval:.2e})")
print(f"Drift detected: {pred_pval < 0.01}")

In [ ]:
# Save monitoring results to Snowflake
monitoring_records = []
run_date = datetime.now()

# Performance metrics
for metric in ['precision', 'recall', 'f1', 'auc_roc']:
    train_val = training_metrics.get(metric, training_metrics.get('f1_score'))
    if metric == 'f1': train_val = training_metrics['f1_score']
    monitoring_records.append({
        'RUN_DATE': run_date, 'METRIC_NAME': f'perf_{metric}',
        'TRAINING_VALUE': float(train_val), 'CURRENT_VALUE': float(new_metrics[metric]),
        'DRIFT_SCORE': float(abs(new_metrics[metric] - train_val)),
        'ALERT_FLAG': metric == 'f1' and new_metrics[metric] < f1_threshold
    })

# Data drift
for _, row in drift_df.iterrows():
    monitoring_records.append({
        'RUN_DATE': run_date, 'METRIC_NAME': f'drift_{row["feature"]}',
        'TRAINING_VALUE': 0.0, 'CURRENT_VALUE': float(row['ks_stat']),
        'DRIFT_SCORE': float(row['ks_stat']),
        'ALERT_FLAG': bool(row['drift_detected'])
    })

# Prediction drift
monitoring_records.append({
    'RUN_DATE': run_date, 'METRIC_NAME': 'prediction_drift_ks',
    'TRAINING_VALUE': 0.0, 'CURRENT_VALUE': float(pred_ks),
    'DRIFT_SCORE': float(pred_ks), 'ALERT_FLAG': pred_pval < 0.01
})

mon_df = pd.DataFrame(monitoring_records)
session.create_dataframe(mon_df).write.mode("overwrite").save_as_table(
    "COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG"
)
print(f"Monitoring log written: {len(mon_df)} records")
print(f"Alerts triggered: {mon_df['ALERT_FLAG'].sum()}")
print(mon_df[['METRIC_NAME','TRAINING_VALUE','CURRENT_VALUE','DRIFT_SCORE','ALERT_FLAG']].to_string(index=False))

### Automated Monitoring: Snowflake Task & Alert

Create a scheduled stored procedure that runs daily to:
1. Score new transactions
2. Compute drift metrics
3. Write to monitoring log

An Alert fires when drift exceeds threshold.

In [ ]:
%%sql -r create_mon_table
CREATE OR REPLACE TABLE COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG (
    RUN_DATE TIMESTAMP_NTZ,
    METRIC_NAME VARCHAR,
    TRAINING_VALUE FLOAT,
    CURRENT_VALUE FLOAT,
    DRIFT_SCORE FLOAT,
    ALERT_FLAG BOOLEAN
)

In [ ]:
%%sql -r create_proc
CREATE OR REPLACE PROCEDURE COCO_DS_DEMO.PUBLIC.RUN_MODEL_MONITORING()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET v_run_date TIMESTAMP := CURRENT_TIMESTAMP();
    LET v_baseline_fraud_rate FLOAT := 0.03;
    LET v_baseline_avg_amount FLOAT := 82.0;
    
    LET v_current_fraud_rate FLOAT := (
        SELECT AVG(CASE WHEN PREDICTED_FRAUD THEN 1.0 ELSE 0.0 END)
        FROM COCO_DS_DEMO.PUBLIC.FRAUD_PREDICTIONS
    );
    LET v_current_avg_prob FLOAT := (
        SELECT AVG(FRAUD_PROBABILITY)
        FROM COCO_DS_DEMO.PUBLIC.FRAUD_PREDICTIONS
    );
    LET v_current_avg_amount FLOAT := (
        SELECT AVG(AMOUNT) FROM COCO_DS_DEMO.PUBLIC.TRANSACTIONS
    );
    
    LET v_rate_drift FLOAT := ABS(:v_current_fraud_rate - :v_baseline_fraud_rate);
    LET v_amount_drift FLOAT := ABS(:v_current_avg_amount - :v_baseline_avg_amount) / :v_baseline_avg_amount;
    LET v_prob_drift FLOAT := ABS(:v_current_avg_prob - 0.03);
    LET v_rate_alert BOOLEAN := (:v_rate_drift > 0.02);
    LET v_prob_alert BOOLEAN := (:v_prob_drift > 0.05);
    LET v_amount_alert BOOLEAN := (:v_amount_drift > 0.2);
    
    INSERT INTO COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG
        (RUN_DATE, METRIC_NAME, TRAINING_VALUE, CURRENT_VALUE, DRIFT_SCORE, ALERT_FLAG)
    VALUES (:v_run_date, 'predicted_fraud_rate', :v_baseline_fraud_rate, :v_current_fraud_rate, :v_rate_drift, :v_rate_alert);
    
    INSERT INTO COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG
        (RUN_DATE, METRIC_NAME, TRAINING_VALUE, CURRENT_VALUE, DRIFT_SCORE, ALERT_FLAG)
    VALUES (:v_run_date, 'avg_fraud_probability', 0.03, :v_current_avg_prob, :v_prob_drift, :v_prob_alert);
    
    INSERT INTO COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG
        (RUN_DATE, METRIC_NAME, TRAINING_VALUE, CURRENT_VALUE, DRIFT_SCORE, ALERT_FLAG)
    VALUES (:v_run_date, 'amount_distribution_drift', :v_baseline_avg_amount, :v_current_avg_amount, :v_amount_drift, :v_amount_alert);
    
    RETURN 'Monitoring complete at ' || :v_run_date::VARCHAR;
END;

In [ ]:
%%sql -r create_task
-- Create daily monitoring task
CREATE OR REPLACE TASK COCO_DS_DEMO.PUBLIC.DAILY_MODEL_MONITORING_TASK
    WAREHOUSE = COCO_DS_WH
    SCHEDULE = 'USING CRON 0 6 * * * UTC'
    COMMENT = 'Daily fraud model monitoring'
AS
    CALL COCO_DS_DEMO.PUBLIC.RUN_MODEL_MONITORING();

ALTER TASK COCO_DS_DEMO.PUBLIC.DAILY_MODEL_MONITORING_TASK RESUME

In [ ]:
%%sql -r create_alert
-- Alert fires when drift detected
CREATE OR REPLACE ALERT COCO_DS_DEMO.PUBLIC.MODEL_DRIFT_ALERT
    WAREHOUSE = COCO_DS_WH
    SCHEDULE = 'USING CRON 30 6 * * * UTC'
    IF (EXISTS (
        SELECT 1 FROM COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG
        WHERE RUN_DATE > DATEADD('hour', -2, CURRENT_TIMESTAMP())
          AND ALERT_FLAG = TRUE
    ))
    THEN
        CALL SYSTEM$SEND_EMAIL(
            'model_monitoring_integration',
            'ml-team@company.com',
            'ALERT: Fraud Model Drift Detected',
            'Drift or performance degradation detected. Check MODEL_MONITORING_LOG.'
        )

In [ ]:
%%sql -r test_monitoring
-- Test the procedure
CALL COCO_DS_DEMO.PUBLIC.RUN_MODEL_MONITORING();

-- Verify log
SELECT * FROM COCO_DS_DEMO.PUBLIC.MODEL_MONITORING_LOG
ORDER BY RUN_DATE DESC
LIMIT 10

In [ ]:
# Final summary
print("=" * 60)
print("  MLOps Pipeline Summary")
print("=" * 60)
print(f"\n{'Dataset:':<28} COCO_DS_DEMO.PUBLIC.TRANSACTIONS")
print(f"{'Rows:':<28} 1,000,000 | Fraud Rate: ~3%")
print(f"\n{'Best Model:':<28} {best_name}")
print(f"{'F1 Score:':<28} {results[best_name]['f1']:.4f}")
print(f"{'AUC-ROC:':<28} {results[best_name]['auc_roc']:.4f}")
print(f"\n{'Model Registry:':<28} fraud_detector v1")
print(f"{'Predictions Table:':<28} FRAUD_PREDICTIONS")
print(f"{'Monitoring Log:':<28} MODEL_MONITORING_LOG")
print(f"\n{'Monitoring Task:':<28} DAILY_MODEL_MONITORING_TASK (6:00 AM UTC)")
print(f"{'Drift Alert:':<28} MODEL_DRIFT_ALERT (6:30 AM UTC)")
print(f"{'Drift Features Detected:':<28} {drift_df['drift_detected'].sum()} / {len(drift_df)}")
print("\n" + "=" * 60)
print("  Pipeline Complete!")
print("=" * 60)